# AI Lab - Practical Question No. 30
## 8-Puzzle Solver using A* (Manhattan Distance Heuristic)

| Specification | Detail |
| :--- | :--- |
| **Title** | 8-Puzzle Solver using A* (Manhattan) |
| **Syllabus Area** | Informed Search and Heuristics |
| **Difficulty** | 🔴 Expert |
| **Practical Type** | Integrated Problem |
| **Core AI Concept** | A* on complex state spaces (181,440 reachable states, branching factor $\approx 3$) |
| **Evaluation Function** | $f(n) = g(n) + h_{\text{Manhattan}}(n)$ |
| **Python Concepts Tested** | 1D tuple fast hashing, deepcopy avoidance, binary min-heap via `heapq` |
| **Complexity Expectation** | High (memory limits reached if poorly optimized) |
| **Sample Input** | `[1,2,3, 4,0,5, 6,7,8]` |
| **Sample Output** | `Solved in 2 moves: [Board states...]` |
| **Viva Topics** | • Inversions to check solvability<br>• State hashing importance |
| **Optional Challenge** | Mathematically verify if the board is solvable before searching |

## 1. Theoretical Foundations: State Space & Evaluation Function

### State Space Scale & Invariant Parity
The 8-puzzle consists of a $3 \times 3$ grid containing 8 numbered tiles ($1..8$) and an empty space ($0$).
- **Total Permutations**: $9! = 362,880$.
- **Reachable States**: Exactly half of these, $9! / 2 = 181,440$ states, are reachable from any valid configuration.
- The states are partitioned into two disjoint equivalence classes by the **parity of inversions** in the symmetric group $S_9$.

### Evaluation Function: $f(n) = g(n) + h(n)$
- $g(n)$: The exact path cost (number of tile sliding moves) incurred from the initial state to state $n$.
- $h(n)$: The Manhattan distance heuristic ($L_1$ norm), summing the grid offsets of all numbered tiles strictly ignoring the blank ($0$):
  $$h_{\text{Manhattan}}(n) = \sum_{\text{tile}=1}^{8} \left(|r_{\text{curr}}(\text{tile}) - r_{\text{goal}}(\text{tile})| + |c_{\text{curr}}(\text{tile}) - c_{\text{goal}}(\text{tile})|\right)$$
- $f(n)$: The estimated total cost of the optimal solution passing through state $n$.

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is accessible
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.a_star_eight_puzzle import (
    CLASSIC_GOAL_STATE,
    SAMPLE_GOAL_STATE_2MOVE,
    SAMPLE_INPUT_STATE,
    TWO_MOVE_TO_CLASSIC_STATE,
    a_star_eight_puzzle,
    compare_search_heuristics,
    count_inversions,
    explain_solvability,
    get_eight_puzzle_viva_reference,
    is_solvable,
    manhattan_distance,
    parse_puzzle_state,
    render_board_ascii,
    render_step_by_step_boards,
)

print("Module successfully imported!")
print("Sample Input State:", SAMPLE_INPUT_STATE)
print("Classic Goal State:", CLASSIC_GOAL_STATE)
print("2-Move Target State:", SAMPLE_GOAL_STATE_2MOVE)

## 2. Mathematical Solvability Verification (Optional Challenge)

### Inversion Parity Theorem
An **inversion** is any pair of numbered tiles $(i, j)$ such that $i < j$ in linear row-major order, but $A[i] > A[j]$ (ignoring $0$).
- Horizontal moves change inversion count by $0$.
- Vertical moves jump a tile over exactly $2$ other tiles, changing inversion count by $+2, 0,$ or $-2$.
- Therefore, $\Delta \text{inv} \equiv 0 \pmod 2$.
- A board is solvable if and only if $\text{inv}(\text{initial}) \pmod 2 == \text{inv}(\text{goal}) \pmod 2$.

In [ ]:
sample_state = (1, 2, 3, 4, 0, 5, 6, 7, 8)
unsolvable_state = (1, 2, 3, 4, 5, 6, 8, 7, 0)

solv1, reason1, inv1, _ = explain_solvability(sample_state, CLASSIC_GOAL_STATE)
print(f"Sample State (Inversions: {inv1}): Solvable? {solv1}")
print(reason1)
print("-" * 60)

solv2, reason2, inv2, _ = explain_solvability(unsolvable_state, CLASSIC_GOAL_STATE)
print(f"Modified State (Inversions: {inv2}): Solvable? {solv2}")
print(reason2)

## 3. Running A* Search with Manhattan Distance

Let us solve the canonical syllabus sample input `[1,2,3, 4,0,5, 6,7,8]` to target `[1,2,3, 4,5,8, 6,7,0]`.

In [ ]:
# Solve syllabus sample input (2-move target)
result_2move = a_star_eight_puzzle(
    initial_state=SAMPLE_INPUT_STATE,
    goal_state=SAMPLE_GOAL_STATE_2MOVE,
    heuristic="manhattan",
)

# Print exact syllabus sample output
print(result_2move.format_sample_output())
print("\nDetailed Step-by-Step Visualization:")
print(result_2move.format_detailed_solution(use_ascii=True))

## 4. Solving to Classic Goal (1-8, 0)

Now let us solve the same starting configuration to reach the canonical classic goal state `(1, 2, 3, 4, 5, 6, 7, 8, 0)`.

In [ ]:
result_classic = a_star_eight_puzzle(
    initial_state=SAMPLE_INPUT_STATE,
    goal_state=CLASSIC_GOAL_STATE,
    heuristic="manhattan",
)

print(f"Solved in {result_classic.move_count} optimal moves!")
print(f"Nodes expanded: {result_classic.nodes_expanded}")
print(f"Execution time: {result_classic.time_taken * 1000:.2f} ms")
print(f"Sequence of moves:\n", " -> ".join(result_classic.moves))

## 5. Comparative Evaluation: Manhattan vs. Misplaced Tiles vs. BFS

Here we compare the performance of A* with Manhattan Distance ($h_2$), A* with Misplaced Tiles ($h_1$), and unguided BFS ($h=0$).

In [ ]:
test_board = (1, 2, 3, 0, 4, 6, 7, 5, 8)
comparison = compare_search_heuristics(test_board, CLASSIC_GOAL_STATE, include_bfs=True)
print(comparison.format_table())

## 6. University Viva Voce Reference Guide

In [ ]:
print(get_eight_puzzle_viva_reference())